# One column is a majority vote

For a column with two filled values, write $t$ for the transported count and $s$ for the stayed count. The share transported sits above one half exactly when $t > s$:

$$
\dfrac{t}{t + s} - \dfrac{1}{2}
= \dfrac{2t - (t + s)}{2(t + s)}
= \dfrac{t - s}{2(t + s)}.
$$

The denominator is positive. The sign is the sign of $t - s$. Predicting the majority class is the same act as predicting transported when the odds $t/s$ are greater than 1.

Cryosleep true: $t = 2483$, $s = 554$.

$$
\dfrac{2483}{554} > 1
\qquad\text{because}\qquad
2483 - 554 = 1929.
$$

Cryosleep false: $t = 1789$, $s = 3650$.

$$
\dfrac{1789}{3650} < 1
\qquad\text{because}\qquad
1789 - 3650 = -1861.
$$

A logistic regression that uses only this one binary column, and matches these two rates, draws its decision at one half. That decision is the cryosleep rule. Fitting a coefficient cannot beat the table on a column that has already been split into its own cells.

The blank cryosleep cell is a third level: $t = 106$, $s = 111$, and $106 - 111 = -5$. The majority is "stayed," by five passengers. The cryosleep rule already says that.


In [2]:
# Locate the contest tables beside this notebook, or under the course root.
from pathlib import Path
import csv

def manifest_path(name):
    here = Path.cwd()
    for folder in [here, *here.parents]:
        nested = folder / "14-Spaceship Titanic" / "data" / name
        direct = folder / "data" / name
        if nested.exists():
            return nested
        if direct.exists() and (folder / "data" / "sample_submission.csv").exists():
            return direct
    raise FileNotFoundError(name)

def read_manifest(name):
    with manifest_path(name).open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))

train = read_manifest("train.csv")
test = read_manifest("test.csv")
submission = read_manifest("sample_submission.csv")
print(len(train), len(test), len(submission))
print(train[0]["PassengerId"], train[0]["Name"])


8693 4277 4277
0001_01 Maham Ofracculy


In [3]:
# The sign of t - s is the hard prediction.
from fractions import Fraction
cells = {"True": (2483, 554), "False": (1789, 3650), "": (106, 111)}
for flag, (transported, stayed) in cells.items():
    gap = transported - stayed
    odds = Fraction(transported, stayed)
    print(repr(flag), gap, odds, odds > 1)
assert 2483 - 554 == 1929
assert 1789 - 3650 == -1861
assert 106 - 111 == -5
assert Fraction(2483, 3037) - Fraction(1, 2) == Fraction(1929, 2 * 3037)
assert Fraction(1789, 5439) - Fraction(1, 2) == Fraction(-1861, 2 * 5439)
identity = Fraction(2483 - 554, 2 * (2483 + 554))
assert identity == Fraction(1929, 6074)


'True' 1929 2483/554 True
'False' -1861 1789/3650 False
'' -5 106/111 False


## Pitfall

Odds of $2483/554$ are not a license to invent a decimal probability and then round it. The hard prediction uses only the sign of $t - s$. The fraction $2483/3037$ is already the rate.

## Summary

On one categorical column, the optimal hard rule is the majority inside each cell. For cryosleep, that rule is the 6244-row rule. A one-column model repeats it.
